In [ ]:
!pip install bitsandbytes transformers peft accelerate trl datasets lm-eval

### Импорты + сид

In [ ]:
import os
import random
import numpy as np
import torch
import torch.profiler
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)

seed_everything(322)

### Загрузка датасета + модели

По большей части датасет про то как следовать инструкциям на урусском языке. Там формат что есть вопросы/задачи и к каждой дается идеальный ответ и учитывая что делал Илья Гусев, насчет его качества сомневаться я не буду

In [ ]:
dataset_id = "IlyaGusev/ru_turbo_alpaca"
dataset = load_dataset(dataset_id, revision="refs/convert/parquet")

def format_dataset(example):
    example["text"] = (
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return example

formatted_dataset = dataset.map(format_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Решил взять Qwen, так как это база. Очень нравятся все семейства квенов. По идее можно было взять еще и llama, но как-то не.

Еше учитывая специфику датасета, можно было бы взять не `Instruct` версию, а `Chat`

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

### Генерация текста до обучения + benchmark

In [ ]:
test_prompts =[
    "<|im_start|>user\nНапиши стихотворение про ежа и молоко.<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>user\nЧто такое Python?<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>user\nПочему 2+2 = 4?<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>user\nНапиши рецепт пиццы с пепперони.<|im_end|>\n<|im_start|>assistant\n"
]

model.eval()

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            pad_token_id=tokenizer.eos_token_id
        )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("-" * 40)

user
Напиши стихотворение про ежа и молоко.
assistant
Вот простое стихотворение про ежа и молоко:

Ежик, ежик,
В корзинке лежит молоко.
Мясо-мясо,
Он его в рот набирает.

Корзинка полна,
Сытый ежик смеется.
А молоко на кухоньке,
Ждет своей удачи.
----------------------------------------
user
Что такое Python?
assistant
Python - это высокопроизводительный язык программирования, который был разработан в 1980-х годах и создан Джеймсом Робертсоном. Он получил название от популярной болгарской музыки.

Основные черты Python:
1. Простой и читаемый код.
2. Быстрый исполнение (разработка и выполнение кода занимает меньше времени).
3. Оптимизация производительности благодаря использованию стека для хранения данных.
4. Возможность использования функций на уровне
----------------------------------------
user
Почему 2+2 = 4?
assistant
Формула "2 + 2 = 4" основана на математических принципах и утверждена множеством исследований. Вот несколько ключевых аспектов, объясивающих эту формулу:

1. Множест

Для рецепта пиццы не хватило `max_tokens`, но наверное не страшно.

Самый ужас это стих, мне прям плохо стало от него

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen2.5-1.5B-Instruct,dtype=float16 \
    --tasks xnli_ru \
    --device cuda \
    --batch_size 4

2026-04-22:09:35:58 INFO     [_cli.run:376] Selected Tasks: ['xnli_ru']
2026-04-22:09:35:58 WARNING  [evaluator:181] pretrained=Qwen/Qwen2.5-1.5B-Instruct appears to be an instruct or chat variant but chat template is not applied. Recommend setting
        `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-04-22:09:36:00 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-22:09:36:00 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'float16'}
2026-04-22:09:36:06 INFO     [models.huggingface:161] Using device 'cuda'
2026-04-22:09:36:07 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
Loading weights: 100% 338/338 [00:07<00:00, 48.17it/s, Materializing param=model.norm.weight]
README.md: 20.8kB [00:00, 37.0MB/s]
ru/train

Датасет у нас про логическое следование по текстам. И метрика выглядит как-то печально прям

### Обучение

Адаптеры добавлены для всех линейных слоев, что должно чуть-чуть увеличить время обучения, но взамен дать больше качества

батч небольшой по дефолту + еще накопление градиентов используется

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

training_args = SFTConfig(
    output_dir="./qwen_ru_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    save_steps=50,
    max_length=256,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

trainer.model.save_pretrained("qwen_ru_lora")
tokenizer.save_pretrained("qwen_ru_lora")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,1.451737
20,1.198034
30,1.184644
40,1.167035
50,1.139573
60,1.158973
70,1.116961
80,1.134386
90,1.167423
100,1.133073


('qwen_ru_lora/tokenizer_config.json',
 'qwen_ru_lora/chat_template.jinja',
 'qwen_ru_lora/tokenizer.json')

### Генерация после дообучения + benchmark

In [ ]:
model.config.use_cache = True
model.eval()

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            pad_token_id=tokenizer.eos_token_id
        )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("-" * 40)

user
Напиши стихотворение про ежа и молоко.
assistant
Ежик на лугу сидит,
С керосиновым ножом в руке.
Молоко, которое он съел,
На его хвосте осталось.
----------------------------------------
user
Что такое Python?
assistant
Python - это язык программирования, который был разработан для быстрого создания интерактивных приложений. Он используется в различных областях, таких как научные исследования, инженерия и финансы. Python имеет простую структуру и высокую производительность, что делает его популярным среди начинающих и опытных программистов.
----------------------------------------
user
Почему 2+2 = 4?
assistant
Сложение чисел 2 и 2 равно 4, потому что это закономерность в математике. Это свойство является основным правилом сложения, которое мы научились изучать в школе.
----------------------------------------
user
Напиши рецепт пиццы с пепперони.
assistant
1. Нарежь мясо на кусочки и обжарь их в масле до золотистого цвета.
2. Нарежь сыр на кубики или порции.
3. На противень вылож

Стих все еще ужасен, причем тут керосиновый нож даже знать не хочу...

Зато есть рецепт пиццы!!!

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen2.5-1.5B-Instruct,peft=qwen_ru_lora,dtype=float16 \
    --tasks xnli_ru \
    --device cuda \
    --batch_size 4

2026-04-22:09:51:24 INFO     [_cli.run:376] Selected Tasks: ['xnli_ru']
2026-04-22:09:51:24 WARNING  [evaluator:181] pretrained=Qwen/Qwen2.5-1.5B-Instruct appears to be an instruct or chat variant but chat template is not applied. Recommend setting
        `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-04-22:09:51:26 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-22:09:51:26 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-1.5B-Instruct', 'peft': 'qwen_ru_lora', 'dtype': 'float16'}
2026-04-22:09:51:32 INFO     [models.huggingface:161] Using device 'cuda'
2026-04-22:09:51:34 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
Loading weights: 100% 338/338 [00:06<00:00, 50.99it/s, Materializing param=model.norm.weight]
2026-04-22:09:51:42 

Видимо из-за того что мы дали датасет в котором основной посыл корректно следовать инструкциям, то наша модель немного отупела в логике

### Профайлинг

In [ ]:
sample_dataloader = trainer.get_train_dataloader()
batch = next(iter(sample_dataloader))

batch = {k: v.to("cuda") for k, v in batch.items() if k in["input_ids", "attention_mask", "labels"]}

model.train()

with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
    record_shapes=True,
    profile_memory=True,
    with_stack=True
) as prof:

    for step in range(5):
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        prof.step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us        4.838s       130.81%        4.838s        1.613s           0 B           0 B           0 B           0 

Топ 1 и 2 на CPU это операции для нашей QLoRa, то есть тут видно что из за постоянной распаковки нашей модели из 4 бит, считаем данные и потом удаляем память

То есть большой нюанс обучения QLoRa проявляется вот тут, на операциях в CPU, где мы экономим на VRAM, но зато долго готовим данные